# CT-RATE → CT-CLIP embeddings → Drive  (disk-safe, resumable)

**Changed from a raw-volume downloader to an embedding extractor.** This now behaves
exactly like `ctclip_features_colab.ipynb`: it **streams one volume → CT-CLIP encode →
saves the 512-d vector to Drive → deletes the volume**, so peak disk stays < ~1 GB no
matter how big the split is.

- **Cell 1** encodes **all VALIDATION** volumes.
- **Cell 2** encodes **all TRAINING** volumes.

Both write `.pt` files into the **same cache as the other notebook**:
`MyDrive/3dCT/ctclip_cache/img/`. The loop **skips any volume already cached**, so it
never redoes the old 600-pair examples and is fully **resumable** — just re-run a cell
after a Colab timeout.

**Prereqs:** GPU runtime (A100/L4, ≥24 GB ideal); accept CT-RATE + CT-CLIP licenses on
Hugging Face; have a read token.


In [ ]:
# --- GPU check ---
!nvidia-smi || echo 'NO GPU — Runtime > Change runtime type > GPU (A100/L4, >=24 GB)'
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


In [ ]:
# --- clone CT-CLIP + install (CTViT is torch-picky; if imports fail, restart once) ---
%cd /content
![ -d CT-CLIP ] || git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git
%cd /content/CT-CLIP
!pip install -q -e transformer_maskgit
!pip install -q -e CT_CLIP
!pip install -q nibabel scipy huggingface_hub transformers tqdm
%cd /content
import sys
for p in ['/content/CT-CLIP/CT_CLIP', '/content/CT-CLIP/transformer_maskgit']:
    if p not in sys.path:
        sys.path.insert(0, p)
import ct_clip, transformer_maskgit
print('ct_clip ->', ct_clip.__file__)


In [ ]:
# --- Drive + repo + auth + weights + embedder + define e-
import os, time, json
from google.colab import drive
drive.mount('/content/drive')

# SAME cache directory as ctclip_features_colab.ipynb  (do not change)
DRIVE_OUT = '/content/drive/MyDrive/3dCT/ctclip_cache'
IMG_DIR   = os.path.join(DRIVE_OUT, 'img')
os.makedirs(IMG_DIR, exist_ok=True)
TMP = '/content/_vol_tmp'; os.makedirs(TMP, exist_ok=True)

# get our repo helper (ctclip_utils.py)
REPO_URL = 'https://github.com/nprakash1/3dCT.git'
%cd /content
![ -d 3dCT ] || git clone $REPO_URL 3dCT
import sys; sys.path.append('/content/3dCT/scripts')

from huggingface_hub import login, hf_hub_download, HfApi
login()  # paste a READ token (CT-RATE + CT-CLIP must be accepted on HF)
TOKEN = os.environ.get('HF_TOKEN') or True   # True = use the cached login token

from ctclip_utils import (REPO_ID, CTCLIP_WEIGHTS_HF, CTCLIPEmbedder,
                          download_volume)

# CT-CLIP weights (cache to Drive so future sessions reuse them)
WEIGHTS_DIR = '/content/drive/MyDrive/3dCT/ctclip_weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
weights_path = hf_hub_download(REPO_ID, CTCLIP_WEIGHTS_HF, repo_type='dataset',
                               token=TOKEN, local_dir=WEIGHTS_DIR)
emb = CTCLIPEmbedder(weights_path)
print('embedder device:', emb.device, '| cache ->', IMG_DIR)


def list_split_volumes(split):
    """All unique .nii.gz basenames in a CT-RATE split (both '<split>' and
    '<split>_fixed' folders), deduped by filename."""
    api = HfApi()
    files = api.list_repo_files(REPO_ID, repo_type='dataset', token=TOKEN)
    keep = {}
    for f in files:
        if f.endswith('.nii.gz') and (
            f.startswith(f'dataset/{split}/') or f.startswith(f'dataset/{split}_fixed/')):
            keep[os.path.basename(f)] = True
    return sorted(keep)


def encode_split(split):
    """Stream -> CT-CLIP encode -> save 512-d to Drive -> delete. Resumable
    (skips volumes already cached in IMG_DIR). Peak disk < ~1 GB."""
    from tqdm.auto import tqdm
    vols = list_split_volumes(split)
    print(f"{split}: {len(vols)} unique volumes on the Hub")
    new = already = 0; missing = []
    PROG = os.path.join(DRIVE_OUT, f'_progress_{split}.json')
    for i, v in enumerate(tqdm(vols)):
        out_pt = os.path.join(IMG_DIR, v.replace('.nii.gz', '').replace('.nii', '') + '.pt')
        if os.path.exists(out_pt):
            already += 1; continue                      # NO REDO: already cached
        fp = download_volume(v, TOKEN, TMP)             # stream ONE volume
        if not fp:
            missing.append(v); continue
        try:
            e = emb.embed_image_path(fp, normalize=True)     # (1, 512)
            torch.save(e.half().squeeze(0).clone(), out_pt)  # AUTO-SAVE to Drive now
            new += 1
        except Exception as ex:
            print('ENCODE FAIL', v, ex)
        finally:
            try: os.remove(fp)                          # DELETE volume immediately
            except Exception: pass
        if (i + 1) % 10 == 0:
            json.dump({'i': i + 1, 'new': new, 'already': already,
                       'missing': len(missing), 't': time.time()}, open(PROG, 'w'))
    json.dump({'i': len(vols), 'new': new, 'already': already,
               'missing': len(missing)}, open(PROG, 'w'))
    print(f"{split}: newly_encoded={new}  already_cached={already}  "
          f"missing={len(missing)}  ->  {IMG_DIR}")
    if missing:
        print('  missing (re-run to retry):', missing[:10])
    return new, already, missing


In [ ]:
# ===== CELL 1: encode ALL VALIDATION volumes -> Drive (embeddings only) =====
encode_split("valid")


In [ ]:
# ===== CELL 2: encode ALL TRAINING volumes -> Drive (embeddings only) =====
# Big split, but only tiny .pt files are kept; volumes are streamed & deleted.
# Resumable: re-run after any timeout (already-cached volumes are skipped).
encode_split("train")


## Notes

- **Same cache, no redo:** writes to `MyDrive/3dCT/ctclip_cache/img/` (identical to
  `ctclip_features_colab.ipynb`) and skips any `.pt` that already exists — so the old
  600-pair embeddings are reused, not recomputed.
- **Disk-safe:** each volume is downloaded to `/content/_vol_tmp`, encoded, saved, then
  deleted — peak disk < ~1 GB regardless of split size.
- **Resumable:** re-run Cell 1/Cell 2 to continue after a disconnect; progress is logged
  to `_progress_<split>.json` in the cache.
- **Preprocessing** (HU clip −1000/+200, [−1,1], 0.75/0.75/1.5 mm, 480×480×240) is fixed
  inside `ctclip_utils.preprocess_ct` — must match how the trained model expects inputs.
- To alspair dynamic/static **text** embeddings, use section 10 of
  `ctclip_features_colab.ipynb` (kept separate since it needs the labels JSONL).
